# Project: Airline AI Assistant (with Tools)

This notebook builds an AI customer support assistant for a fictional airline,
**FlightAI**, and progressively upgrades it with **tool use** (aka function calling) —
letting the LLM call real Python functions as part of answering a question.

> **Note:** Each `gr.ChatInterface(...).launch()` call opens a local web UI. Only the most
> recently *defined* `chat` function is used by a new `launch()` call, so run cells in order.


## 1. Setup

Import what we need, load the API key, and initialize the client.

Your `.env` file (same folder as this notebook) should contain:

```
OPENAI_API_KEY=sk-...your-key...
```


In [1]:
import os
import json
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


In [2]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - please add OPENAI_API_KEY to your .env file")

MODEL = "gpt-4.1-mini"
openai_client = OpenAI()

# As an alternative, if you'd like to use Ollama instead of OpenAI:
# Check that Ollama is running locally, then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai_client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API Key exists and begins sk-proj-


## 2. A basic airline assistant

Just like earlier chatbot exercises: a system prompt that sets the persona and tone,
and a `chat` function that forwards history + the new message to the OpenAI API.


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""


In [4]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai_client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [5]:
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


## 3. Tools — giving the LLM the ability to call functions

**Tools** (also called "function calling") let you describe a Python function to the
model, so it can decide *when* to call it and *with what arguments*, based on the
conversation. The model never runs your code directly — instead, it tells you
"please call this function with these arguments," you run it yourself, and feed
the result back so the model can finish its answer.

It sounds like the LLM is running code on your machine, but it isn't — **you're
always the one executing the function**, on your own terms. The model only ever
returns a structured request to do so.

### Step 1: write a useful function

We'll start with a simple in-memory lookup: a dictionary of ticket prices by city.


In [6]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [7]:
get_ticket_price("London")


Tool called for city London


'The price of a ticket to London is $799'

### Step 2: describe the function for the LLM

The API needs a specific JSON-like structure describing the function: its name,
what it does, and its parameters (using JSON Schema). This description — not the
actual Python code — is what gets sent to the model.


In [8]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}


In [9]:
# Tools are passed to the API as a list, each wrapped with type "function"
tools = [{"type": "function", "function": price_function}]
tools


[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## 4. Getting OpenAI to actually use our tool

There's a specific request/response dance required to use tools:

1. We call the API with `tools=tools` included.
2. If the model decides it needs the tool, it doesn't answer directly — instead
   `response.choices[0].finish_reason` comes back as `"tool_calls"`, and the message
   contains a `tool_calls` list describing which function to call and with what
   arguments.
3. We run the real Python function ourselves, and package the result as a special
   `"role": "tool"` message.
4. We send the *whole* conversation again — including the model's tool-call request
   and our tool's result — so the model can produce a final natural-language answer.

Here's the updated `chat` function that handles a single tool call:


In [10]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai_client.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content


We also need to write `handle_tool_call`, which reads the model's requested
function name and arguments, actually runs `get_ticket_price`, and formats the
result as a `"tool"` role message (linked back via `tool_call_id` so the model
knows which request this result answers).


In [11]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


Tool called for city London
Tool called for city London


## 5. Handling multiple tool calls, and multiple rounds

Two improvements over the single-tool-call version above:

- **Multiple tool calls in one response** — a user might ask about several cities
  at once ("How much to Paris and Tokyo?"). The model can return *several* entries
  in `message.tool_calls`, so we loop over all of them and build a list of results.
- **Multiple rounds of tool calls** — after we send tool results back, the model
  *might* decide it needs to call another tool before it can finish (rare here, but
  common in more complex agents). Using a `while` loop instead of a single `if`
  handles this correctly: we keep responding to tool calls until the model finally
  returns a normal answer.

First, the version handling multiple tool calls in one round:


In [13]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses


Now the final, most robust version of `chat`: a `while` loop that keeps
resolving tool calls until the model is ready to give a plain-text answer. This is
the version we'll build on for the rest of the notebook.


In [14]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content


In [15]:
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Tool called for city London
Tool called for city Berlin


## 6. Replacing the dictionary with a real database

Right now, ticket prices live in a Python dictionary that resets every time the
notebook restarts. Let's back this with a persistent **SQLite** database instead —
a small step toward a production-ready tool.


In [16]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()


In [17]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"


In [18]:
get_ticket_price("London")  # no data yet - table is empty


DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

We need some way to populate the table. Let's write a `set_ticket_price` helper
and seed the database with a few cities.


In [28]:
def set_ticket_price(city, price):
    print(f"Tool called to update change the price of {city} to {price}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?',
            (city.lower(), price, price)
        )
        conn.commit()


In [20]:
ticket_prices = {"london": 799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)


In [21]:
get_ticket_price("Tokyo")  # now reads from the database


DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [29]:
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for London
Tool called to update change the price of London to 500
DATABASE TOOL CALLED: Getting price for London


## 7. Add a tool to set the price of a ticket

We already have `set_ticket_price(city, price)` as a plain Python function (used
above to seed the database). The exercise is to **expose it as a tool** the LLM
can call directly — so a staff member could say something like *"Set the price
to Rome at $650"* and have the assistant update the database itself.

### Step 1: describe the new tool

Same JSON-Schema style as `price_function`, but with two required parameters
this time: `city` and `price`.


In [30]:
set_price_function = {
    "name": "set_ticket_price",
    "description": "Set (or update) the price of a return ticket to the destination city. "
                    "Use this when someone asks to change, update, or set a ticket price.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city whose ticket price should be set",
            },
            "price": {
                "type": "number",
                "description": "The new ticket price in US dollars",
            },
        },
        "required": ["city", "price"],
        "additionalProperties": False
    }
}


### Step 2: add it to the tools list

Both tools are now available to the model at the same time — it will decide which
one (if any) is relevant based on what the user asks.


In [31]:
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function},
]


### Step 3: extend `handle_tool_calls` to dispatch to either tool

We add a second branch: if the model calls `set_ticket_price`, we pull out `city`
and `price` from the arguments, call our real `set_ticket_price` function, and
return a confirmation message as the tool result.


In [32]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('city')
            price = arguments.get('price')
            set_ticket_price(city, price)
            responses.append({
                "role": "tool",
                "content": f"Ticket price to {city} has been set to ${price}",
                "tool_call_id": tool_call.id
            })
    return responses


### Step 4: try it out

`chat` itself doesn't need to change at all — it already loops over tool calls
generically via `handle_tool_calls`. That's the benefit of the dispatch-by-name
pattern: adding a new tool only means (a) describing it, (b) adding it to the
`tools` list, and (c) adding a branch in the handler.

Try asking things like:
- *"How much is a ticket to Berlin?"*
- *"Set the price to Rome to $650"* followed by *"How much is a ticket to Rome?"*


In [33]:
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


## 8. Business applications

This is a meaningful step beyond a simple Q&A chatbot: the assistant can now **take
actions**, not just answer questions. The same tool-calling pattern used here for
ticket prices generalizes directly to real airline systems — for example, calling
a real booking API to check seat availability, hold a reservation, or issue a
refund, all triggered naturally through conversation.
